# 01 — Extract Text Features

**Issue #4** — Preprocess metadata, apply TF-IDF vectorization and LDA topic modeling, and save results to `data/processed/text_features.pkl`.

### Pipeline
1. Load raw metadata (`data/raw/met_data.csv`)
2. Preprocess text fields (title, artist, medium)
3. TF-IDF vectorization
4. LDA topic modeling
5. Metadata feature extraction
6. Save to `data/processed/text_features.pkl`

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import pickle
import logging

logging.basicConfig(level=logging.INFO)

print(f"Project root: {PROJECT_ROOT}")

## 1. Load raw metadata

In [ ]:
data_path = PROJECT_ROOT / "data" / "raw" / "met_data.csv"
df = pd.read_csv(data_path)

print(f"Loaded {len(df)} records")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Check for missing values in text columns
text_cols = ["title", "artist", "medium", "department", "objectDate"]
df[text_cols].isnull().sum()

## 2. Preprocess metadata (title, artist, medium)

In [ ]:
from src.features.text_features import TextFeatureExtractor

extractor = TextFeatureExtractor(
    max_features=500,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
)

# Combine text fields into a single column
combined_text = extractor.combine_text_fields(df, columns=["title", "artist", "medium"])

print("Sample combined texts:")
for i in range(3):
    print(f"  [{i}] {combined_text.iloc[i][:120]}...")

In [ ]:
# Preview preprocessing on a few samples
print("Preprocessing examples:")
for i in range(3):
    raw = combined_text.iloc[i]
    clean = extractor.preprocess_text(raw)
    print(f"  Raw:   {raw[:80]}")
    print(f"  Clean: {clean[:80]}")
    print()

## 3. TF-IDF vectorization

In [ ]:
tfidf_features = extractor.extract_tfidf_features(combined_text, fit=True)

print(f"TF-IDF matrix shape: {tfidf_features.shape}")
print(f"  {tfidf_features.shape[0]} documents x {tfidf_features.shape[1]} features")

In [ ]:
# Top TF-IDF terms
feature_names = extractor.tfidf_vectorizer.get_feature_names_out()
mean_tfidf = tfidf_features.mean(axis=0)
top_indices = mean_tfidf.argsort()[-15:][::-1]

print("Top 15 TF-IDF terms (by mean score):")
for idx in top_indices:
    print(f"  {feature_names[idx]:30s} {mean_tfidf[idx]:.4f}")

## 4. LDA topic modeling

In [ ]:
N_TOPICS = 10

topic_features = extractor.extract_topic_features(tfidf_features, n_topics=N_TOPICS, fit=True)

print(f"Topic distribution shape: {topic_features.shape}")
print(f"  {topic_features.shape[0]} documents x {topic_features.shape[1]} topics")

In [ ]:
# Display top words per topic
top_words = extractor.get_top_words_per_topic(n_words=8)

print("LDA Topics:")
for topic_id, words in top_words.items():
    print(f"  Topic {topic_id:2d}: {', '.join(words)}")

## 5. Metadata features

In [ ]:
metadata_features = extractor.extract_metadata_features(df)

print(f"Metadata features shape: {metadata_features.shape}")
print(f"Columns: {list(metadata_features.columns)}")
metadata_features.head()

## 6. Save to `data/processed/text_features.pkl`

In [ ]:
text_features = {
    "tfidf": tfidf_features,
    "topics": topic_features,
    "metadata": metadata_features.values,
    "feature_names": list(feature_names),
    "topic_words": top_words,
    "metadata_columns": list(metadata_features.columns),
    "extractor": extractor,
}

save_path = PROJECT_ROOT / "data" / "processed" / "text_features.pkl"
save_path.parent.mkdir(parents=True, exist_ok=True)

with open(save_path, "wb") as f:
    pickle.dump(text_features, f)

print(f"Saved text features to {save_path}")
print(f"  File size: {save_path.stat().st_size / 1024:.1f} KB")

## Summary

In [ ]:
print("=== Text Feature Extraction Summary ===")
print(f"Records processed:   {len(df)}")
print(f"TF-IDF features:     {tfidf_features.shape[1]}")
print(f"LDA topics:          {topic_features.shape[1]}")
print(f"Metadata features:   {metadata_features.shape[1]}")
print(f"Output saved to:     data/processed/text_features.pkl")